## Imports

In [1]:
import os
import dagshub
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing


In [3]:
#dagshub.init(repo_owner="UBGidado", repo_name="Distributed-Computing-System", mlflow=True)

mlflow.set_tracking_uri("http://127.0.0.1:5000")  # local tracking
#mlflow.set_tracking_uri("https://dagshub.com/UBGidado/Distributed-Computing-System.mlflow")  # DagsHub remote tracking

mlflow.set_experiment("California Housing ML Experiments")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: http://127.0.0.1:5000


## Load and Prepare Dataset

In [4]:
data = fetch_california_housing(as_frame=True)
df = data.frame


# Split dataset
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")


Train: (14448, 8), Validation: (3096, 8), Test: (3096, 8)


## Function for Experiment Logging

In [6]:
def run_experiment(model_name, model, X_train, X_val, y_train, y_val):
    with mlflow.start_run(run_name=model_name):
        # Train model
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        r2 = r2_score(y_val, preds)

        # 📒 Log parameters, metrics, and model
        mlflow.log_param("model_name", model_name)
        if hasattr(model, "get_params"):
            mlflow.log_params(model.get_params())
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.sklearn.log_model(model, "model")

        #  Feature Importance or Coefficients
        plt.figure(figsize=(8, 5))
        if hasattr(model, "feature_importances_"):
            importance = model.feature_importances_
            feature_names = X_train.columns
            plt.barh(feature_names, importance)
            plt.title(f"{model_name} Feature Importance")
        elif hasattr(model, "coef_"):
            coef = model.coef_
            feature_names = X_train.columns
            plt.barh(feature_names, coef)
            plt.title(f"{model_name} Coefficients")
        else:
            plt.text(0.5, 0.5, "No feature importance available", ha="center")
        plt.tight_layout()
        plt.savefig("feature_importance.png")
        mlflow.log_artifact("feature_importance.png")
        plt.close()

        #Residual Plot
        residuals = y_val - preds
        plt.figure(figsize=(6, 4))
        plt.scatter(preds, residuals, alpha=0.6)
        plt.axhline(y=0, color="red", linestyle="--")
        plt.title(f"{model_name} Residual Plot")
        plt.xlabel("Predicted")
        plt.ylabel("Residuals")
        plt.tight_layout()
        plt.savefig("residual_plot.png")
        mlflow.log_artifact("residual_plot.png")
        plt.close()

        #  Artifact: Predictions CSV
        
        pred_df = pd.DataFrame({
            "y_true": y_val.values,
            "y_pred": preds
        })
        pred_df.to_csv("predictions.csv", index=False)
        mlflow.log_artifact("predictions.csv")

        print(f"{model_name} -> RMSE: {rmse:.4f}, R2: {r2:.4f}")
    return rmse, r2


## Run Experiments (Models 1–4)

In [7]:

run_experiment("LinearRegression", LinearRegression(), X_train, X_val, y_train, y_val)

run_experiment("RandomForest", RandomForestRegressor(n_estimators=100, random_state=42), X_train, X_val, y_train, y_val)

run_experiment("XGBoost", xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42), X_train, X_val, y_train, y_val)

run_experiment("GradientBoosting", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42), X_train, X_val, y_train, y_val)


2025/10/07 16:05:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/07 16:05:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


LinearRegression -> RMSE: 0.7354, R2: 0.5848
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/297070328126627129/runs/ed2ea131a3d54888927f959cef2ea87d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/297070328126627129


2025/10/07 16:06:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/07 16:06:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RandomForest -> RMSE: 0.5196, R2: 0.7927
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/297070328126627129/runs/f87d38c1616d446ea6b9a5e3bf1322a8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/297070328126627129


2025/10/07 16:06:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/07 16:06:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost -> RMSE: 0.4887, R2: 0.8167
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/297070328126627129/runs/c86a87464b96437b83a5e97871facf81
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/297070328126627129


2025/10/07 16:06:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/07 16:06:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


GradientBoosting -> RMSE: 0.5554, R2: 0.7632
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/297070328126627129/runs/1c47a24df14e4f03a116bffa472a4ea8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/297070328126627129


(np.float64(0.5554469954889156), 0.7631800757980692)

## Load Best Model

In [9]:
best_run_id = "c86a87464b96437b83a5e97871facf81"

# Load the best model
best_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

# Test prediction
sample = X_test.iloc[:5]
pred = best_model.predict(sample)

print("Sample predictions:\n", pred)


Sample predictions:
 [0.82043505 0.8709467  3.5290236  1.5294279  1.4761605 ]


In [10]:
import joblib
import os

# Save relative to current notebook directory
model_dir = "../mlruns/models"  # or "models" if you create it inside notebooks
os.makedirs(model_dir, exist_ok=True)
joblib.dump(best_model, os.path.join(model_dir, "best_model.pkl"))
print("✅ Model saved to models/best_model.pkl")

✅ Model saved to models/best_model.pkl
